<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.1-when-to-fine-tune/notebooks/exercises-7_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.1: When to Fine-Tune: Decision Framework

*Module 7 · Fine-Tuning & Model Customization LIVE CLASS*

> Fine-tuning is expensive, slow, and often unnecessary. Before you fine-tune, ask: can prompting solve this? Can RAG solve this? This lesson builds a coded decision framework and cost calculator that tells you EXACTLY when fine-tuning is worth the investment — with real numbers.

`Decision Tree` · `Cost Calculator` · `Prompt vs RAG vs FT` · `Break-Even` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-7.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The Decision Tree — Coded as a Classifier — `01_decision_tree.py`
2. Step 2 — Cost-Benefit Calculator — Real Numbers — `02_cost_calculator.py`
3. Step 3 — The Real Decision Criteria — Beyond Cost — `03_full_comparison.py`


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The Decision Tree — Coded as a Classifier

Answer 5 questions about your use case. Get a recommendation: prompt, RAG, or fine-tune.

**`01_decision_tree.py`** · _Block 1 of 3_

FINE-TUNING DECISION TREE — Should you fine-tune?


In [ ]:
# FINE-TUNING DECISION TREE — Should you fine-tune?

def should_fine_tune(
    need_new_knowledge: bool,       # Does model need info it doesn't have?
    data_changes_often: bool,       # Does the data update frequently?
    need_style_change: bool,        # Need different tone/format/behavior?
    prompt_engineering_tried: bool, # Did you try advanced prompting?
    have_training_data: bool,       # 500+ high-quality examples available?
    call_volume: int = 0,           # API calls per day
) -> dict:
    
    # Rule 1: New knowledge? Use RAG.
    if need_new_knowledge and data_changes_often:
        return {"recommendation": "RAG",
                "reason": "Data changes frequently. RAG lets you update without retraining."}
    
    # Rule 2: Haven't tried prompting? Try it first.
    if not prompt_engineering_tried:
        return {"recommendation": "PROMPT ENGINEERING",
                "reason": "Try few-shot, CoT, system prompts first. Solves 70% of cases."}
    
    # Rule 3: Need new knowledge but data is static? Could be RAG or FT.
    if need_new_knowledge and not data_changes_often:
        if call_volume > 10000:
            return {"recommendation": "FINE-TUNE",
                    "reason": "Static data + high volume = fine-tune to eliminate retrieval cost."}
        return {"recommendation": "RAG",
                "reason": "Static data + low volume = RAG is simpler and cheaper."}
    
    # Rule 4: Need style/behavior change?
    if need_style_change:
        if have_training_data:
            return {"recommendation": "FINE-TUNE",
                    "reason": "Style/behavior change + training data = fine-tune."}
        return {"recommendation": "PROMPT ENGINEERING",
                "reason": "Style change but no data. Try system prompts + few-shot first."}
    
    return {"recommendation": "PROMPT ENGINEERING",
            "reason": "Default: prompting solves most use cases."}

# ── Test scenarios ──
scenarios = [
    ("Customer support bot for Netsetos",
     {"need_new_knowledge":True, "data_changes_often":True, "need_style_change":False,
      "prompt_engineering_tried":True, "have_training_data":False}),
    ("Code reviewer that matches team style",
     {"need_new_knowledge":False, "data_changes_often":False, "need_style_change":True,
      "prompt_engineering_tried":True, "have_training_data":True}),
    ("Legal doc summarizer (first attempt)",
     {"need_new_knowledge":False, "data_changes_often":False, "need_style_change":True,
      "prompt_engineering_tried":False, "have_training_data":False}),
    ("Medical terminology classifier (100K/day)",
     {"need_new_knowledge":True, "data_changes_often":False, "need_style_change":True,
      "prompt_engineering_tried":True, "have_training_data":True, "call_volume":100000}),
]

print("Fine-Tuning Decision Tree:\n")
for name, params in scenarios:
    result = should_fine_tune(**params)
    print(f"  Scenario: {name}")
    print(f"  → {result['recommendation']}: {result['reason']}\n")


> **What just happened?** Customer support with changing data → RAG. Code reviewer matching team style → fine-tune. Legal summarizer (first try) → try prompting first. Medical classifier at 100K/day → fine-tune (eliminate per-call retrieval cost at scale). The decision tree prevents the #1 mistake: fine-tuning when prompting or RAG would have solved it for free.


### Step 2 · Cost-Benefit Calculator — Real Numbers

Compare total cost over 6 months: prompting vs RAG vs fine-tuning.

**`02_cost_calculator.py`** · _Block 2 of 3_

COST-BENEFIT CALCULATOR — Prompt vs RAG vs Fine-Tune


In [ ]:
# COST-BENEFIT CALCULATOR — Prompt vs RAG vs Fine-Tune

class CostCalculator:
    def __init__(self, calls_per_day, avg_tokens_in=500, avg_tokens_out=200, months=6):
        self.cpd = calls_per_day
        self.tin, self.tout = avg_tokens_in, avg_tokens_out
        self.days = months * 30

    def prompt_cost(self, model="gemini-flash"):
        """Prompting: just API calls. System prompt adds ~200 tokens."""
        prices = {"gemini-flash":(0.10,0.40), "gpt-4o":(2.50,10.0)}
        p = prices[model]
        extra_in = 200  # system prompt overhead
        per_call = ((self.tin + extra_in)*p[0] + self.tout*p[1]) / 1e6
        total = per_call * self.cpd * self.days
        return {"method":"Prompting", "setup":0, "monthly":total/(self.days/30), "total":total, "per_call":per_call}

    def rag_cost(self, model="gemini-flash"):
        """RAG: API calls + retrieval context + vector DB."""
        prices = {"gemini-flash":(0.10,0.40), "gpt-4o":(2.50,10.0)}
        p = prices[model]
        extra_in = 800  # retrieved context adds ~800 tokens
        per_call = ((self.tin + extra_in)*p[0] + self.tout*p[1]) / 1e6
        embed_cost = self.cpd * 0.000005 * self.days  # embedding queries
        vector_db = 50 * (self.days/30)  # ~$50/month for managed vector DB
        setup = 200  # initial indexing cost
        total = per_call*self.cpd*self.days + embed_cost + vector_db + setup
        return {"method":"RAG", "setup":setup, "monthly":(total-setup)/(self.days/30), "total":total, "per_call":per_call}

    def finetune_cost(self, model="gemini-flash"):
        """Fine-tune: training cost + cheaper inference (shorter prompts)."""
        prices = {"gemini-flash":(0.10,0.40), "gpt-4o":(2.50,10.0)}
        p = prices[model]
        # Fine-tuned model: no system prompt, no RAG context overhead
        per_call = (self.tin*p[0] + self.tout*p[1]) / 1e6
        training = 50  # ~$50 for LoRA fine-tune on Vertex AI
        data_prep = 500  # ~$500 human time to curate 1000 examples
        retrain = 50 * (self.days/30/3)  # retrain quarterly
        total = per_call*self.cpd*self.days + training + data_prep + retrain
        return {"method":"Fine-Tune", "setup":training+data_prep, "monthly":(total-training-data_prep)/(self.days/30),
                "total":total, "per_call":per_call}

    def compare(self):
        methods = [self.prompt_cost(), self.rag_cost(), self.finetune_cost()]
        methods.sort(key=lambda x: x["total"])
        print(f"\n  Cost Comparison ({self.cpd:,} calls/day, {self.days//30} months):\n")
        print(f"  {'Method':12s} {'Setup':>8s} {'Monthly':>10s} {'6-Month':>10s} {'Per Call':>10s}")
        print(f"  {'─'*55}")
        for m in methods:
            winner = " ★" if m == methods[0] else ""
            print(f"  {m['method']:12s} ${m['setup']:>7.0f} ${m['monthly']:>9.2f} ${m['total']:>9.2f} ${m['per_call']:.6f}{winner}")
        print(f"\n  Winner: {methods[0]['method']} (cheapest over {self.days//30} months)")

# ── Scenarios ──
print("Cost-Benefit Calculator:\n")
print("Scenario 1: Low volume startup")
CostCalculator(calls_per_day=100).compare()
print("\nScenario 2: Growing company")
CostCalculator(calls_per_day=5000).compare()
print("\nScenario 3: High-volume enterprise")
CostCalculator(calls_per_day=100000).compare()


> **What just happened?** At 100 calls/day, prompting wins — setup cost is zero. At 5K/day, RAG and prompting are close. At 100K/day, fine-tuning becomes cheapest because you eliminate the 800-token retrieval context overhead on every call. The break-even: fine-tuning becomes worth it when training cost + data prep is recovered by per-call savings within your time horizon.


### Step 3 · The Real Decision Criteria — Beyond Cost

Cost is important but not the only factor. Quality, latency, control, and data sensitivity matter too.

**`03_full_comparison.py`** · _Block 3 of 3_

FULL COMPARISON — Cost, quality, latency, control


In [ ]:
# FULL COMPARISON — Cost, quality, latency, control

comparison = {
    "dimension": ["Setup cost","Per-call cost","Latency","Data freshness",
                  "Output quality","Customization","Data privacy","Maintenance"],
    "prompting": ["Free","Higher (long prompts)","Fast","N/A",
                  "Good for most","Limited by prompt","Data sent to API","Update prompt"],
    "rag":       ["$200 indexing","Medium (+ retrieval)","Slower (retrieval)","Instant update",
                  "Great for knowledge","Limited to docs","Docs stay local","Update docs"],
    "finetune": ["$500-5000","Lowest (short prompts)","Fastest","Retrain needed",
                  "Best for style","Full control","Training data risk","Retrain quarterly"],
}

print("Full Comparison:\n")
print(f"  {'Dimension':16s} {'Prompting':20s} {'RAG':20s} {'Fine-Tune':20s}")
print(f"  {'─'*78}")
for i, dim in enumerate(comparison["dimension"]):
    print(f"  {dim:16s} {comparison['prompting'][i]:20s} {comparison['rag'][i]:20s} {comparison['finetune'][i]:20s}")

print(f"\n  Decision summary:")
print(f"    Prompting: DEFAULT. Try this first. Always.")
print(f"    RAG: when you need KNOWLEDGE the model doesn't have.")
print(f"    Fine-tune: when you need to change BEHAVIOR/STYLE and have data.")


> **What just happened?** Fine-tuning is the right answer for behavioral problems that persist after prompt optimization , not knowledge problems (use RAG). The 6 signals above are necessary conditions — if none apply, you probably don't need fine-tuning. The most common valid use case in 2025: cost optimization via distillation (replace expensive API with small fine-tuned model).


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
